# GNN-CSP Row/Col Shift Correction Demo

This notebook demonstrates the GNN-CSP pipeline for correcting row/col shifts in table structure recognition.
It uses a Graph Neural Network (GNN) to predict cell relations (Same-Row/Same-Col) and an OR-Tools Constraint Solver (CSP) to enforce structural consistency.

In [ ]:
# 1. Environment Setup
!pip install torch-geometric ortools pandas networkx tqdm

import os
import sys
import torch
import json
import glob
import logging
import numpy as np

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

# Verify Utils
try:
    from gnn_csp_utils import setup_notebook_env, get_pipeline
    setup_notebook_env(PROJECT_ROOT)
    print("Dependencies loaded successfully.")
except ImportError:
    print("Warning: gnn_csp_utils not found, proceeding with manual path setup.")

# Check Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    !nvidia-smi

In [ ]:
# 2. Dataset Configuration

# Define paths to SciTSR dataset
DATA_ROOT = os.path.join(PROJECT_ROOT, "data/external/scitsr/SciTsr_Logical/test/gt")

if not os.path.exists(DATA_ROOT):
    # Fallback to try and look relative to current dir if notebook was moved
    POSSIBLE_ROOTS = [
        "/root/t1-9/data/external/scitsr/SciTsr_Logical/test/gt",
        "../data/external/scitsr/SciTsr_Logical/test/gt"
    ]
    for p in POSSIBLE_ROOTS:
        if os.path.exists(p):
            DATA_ROOT = p
            break

json_files = glob.glob(os.path.join(DATA_ROOT, "*.json"))
print(f"Data Directory: {DATA_ROOT}")
print(f"Found {len(json_files)} JSON files.")

if len(json_files) == 0:
    raise FileNotFoundError("No JSON files found! Check dataset path.")

In [ ]:
# 3. Model & Pipeline Initialization
from hiertable_rag.gnn_csp import GNNCSPPipeline

# Initialize with default weights
pipeline = GNNCSPPipeline(
    device=device,
    semantic_model="roberta-base",
    alpha=0.5, # Visual weight
    beta=0.3,  # Semantic weight
    gamma=0.2  # Layout weight
)
pipeline.gnn.train()
print("Pipeline initialized.")

In [ ]:
# 4. Training (Small Demo Loop)
from torch_geometric.loader import DataLoader
from collections import Counter

# Minimal Dataset Class for formatting
class DemoDataset(torch.utils.data.Dataset):
    def __init__(self, files, graph_builder):
        self.files = files
        self.graph_builder = graph_builder
    def __len__(self): return min(len(self.files), 50) # Limit to 50 for speed
    def __getitem__(self, idx):
        with open(self.files[idx]) as f: data = json.load(f)
        ocr_boxes = [{'box': c.get('box', [0,0,0,0]), 'content': c.get('text', '')} for c in data['cells']]
        # Using random features for demo speed (skip RCA/BERT inference here)
        N = len(ocr_boxes)
        v_feat, s_feat = torch.randn(N, 128), torch.randn(N, 768)
        data_obj, labels = self.graph_builder.build_graph(ocr_boxes, return_labels=True, gt_structure=data)
        # Mock position features
        pos = torch.zeros(N, 4)
        data_obj.x = torch.cat([v_feat, pos, s_feat], dim=-1)
        data_obj.y_edge = labels
        return data_obj

dataset = DemoDataset(json_files, pipeline.graph_builder)
loader = DataLoader(dataset, batch_size=4, shuffle=True)
optimizer = torch.optim.Adam(pipeline.gnn.parameters(), lr=1e-3)

print("Starting demo training...")
for epoch in range(3):
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        if batch.x.shape[0] == 0: continue
        optimizer.zero_grad()
        node_emb, _ = pipeline.gnn(batch)
        logits = pipeline.gnn.predict_edges(node_emb, batch.edge_index)
        if batch.y_edge is not None:
            loss = torch.nn.CrossEntropyLoss()(logits, batch.y_edge)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

# Save trained model
os.makedirs('checkpoints', exist_ok=True)
torch.save(pipeline.gnn.state_dict(), "checkpoints/gnn_demo.pth")
print("Model saved.")

In [ ]:
# 5. Inference & CSP Visualization
pipeline.gnn.eval()
target_file = json_files[0]
with open(target_file) as f: data = json.load(f)

print(f"Running inference on: {target_file}")

ocr_boxes = [{'box': c.get('box', [0,0,0,0]), 'content': c.get('text', '')} for c in data['cells']]
image = torch.zeros(3, 1000, 1000).to(device) # Placeholder image

# Run GNN-CSP
result = pipeline.parse(image, ocr_boxes, use_csp=True, use_semantic=False)

print(f"Recovered Structure: {result['num_rows']} Rows x {result['num_cols']} Cols")
print(f"Constraint Status: {result['constraint_satisfaction']['hard_valid']}")
print(f"Solve Time: {result['solve_time']:.4f}s")

# Compare with GT if available
max_row_gt = max([c.get('row', 0) for c in data['cells']]) + 1
max_col_gt = max([c.get('col', 0) for c in data['cells']]) + 1
print(f"Ground Truth: {max_row_gt} Rows x {max_col_gt} Cols")

if result['num_rows'] == max_row_gt and result['num_cols'] == max_col_gt:
    print("SUCCESS: Structure matches dimensions!")
else:
    print("WARNING: Dimension mismatch (Expected behavior for untrained random features)")